# Week 4 · Notebook 04 — The Twist: How Good Is *Your* Pipeline?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/04_injection_recovery.ipynb)

You found a planet. Great. But suppose you search 1,000 stars and find 10 planets.
**How many did you miss?** Without that number you cannot say anything about how common planets are.

The answer is an experiment: inject fake planets of *known* size and period into *real* data, run your
pipeline **blind**, and count how many come back. The fraction recovered = your **detection efficiency**
(completeness). This is exactly how the Kepler team calibrated their final catalogue.

**Reading:**
- Christiansen et al. 2013 ([ADS](https://ui.adsabs.harvard.edu/abs/2013ApJS..207...35C)) — the first Kepler injection-recovery
- Christiansen et al. 2020, DR25 completeness ([arXiv:2010.04796](https://arxiv.org/abs/2010.04796)) — Sec. 1–3, Fig. of efficiency vs MES
- Hsu et al. 2018 / Burke et al. 2015 — how completeness turns into occurrence rates (skim)

In [ ]:
# --- Setup: run this cell first. Works in Google Colab and locally. ---
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})
print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, inject, plotting
from transitpipe.io import LightCurve
import pandas as pd

## 1. One injection, by hand
Rule #1: **inject into the RAW light curve, before detrending.** Detrending is one of the main
ways pipelines lose planets; injecting after it would hide that loss.

In [ ]:
raw = inject.make_synthetic_lightcurve(seed=5)         # fast, offline; real data in Section 3
fake, truth = inject.inject_transit(raw, period=6.0, rp_rearth=3.0, seed=1)   # then try 2.0 and 1.5!
print({k: round(float(v), 4) for k, v in truth.items()})

PIPELINE = dict(method="median", window_days=1.0)      # freeze YOUR pipeline settings here
flat = detrend.sigma_clip(detrend.flatten(fake, **PIPELINE))
r = search.bls(flat)
print(r.summary())
print("match?", search.is_match(r.period, truth["period"]))

A 3 R⊕ planet is found. Now change `rp_rearth` to **2.0**, then **1.5**, and rerun. At some size the
pipeline starts returning a wrong period: the planet is lost in the noise. One injection tells you
nothing about *how often* that happens. For that you need hundreds — the grid below.

### What counts as "recovered"?
You must decide, and write it in your report. Ours (in `inject._one_trial`):
1. found period within **1%** of the injected one (aliases like 2:1 are logged but *not* counted),
2. found mid-time within one transit duration of an injected transit,
3. **SDE ≥ 7**.

Change any of these and your completeness changes. That's not a bug — it's the point.

## 2. The grid
Draw periods and radii log-uniformly inside each grid cell, inject, search, repeat.

In [ ]:
period_edges = np.geomspace(0.7, 12, 6)      # 5 period bins
radius_edges = np.geomspace(0.7, 6, 6)       # 5 radius bins
df = inject.injection_recovery(raw, period_edges, radius_edges, n_per_cell=8,
                               detrend_kwargs=PIPELINE, bls_kwargs={"oversample": 1.0},
                               n_jobs=os.cpu_count(), progress=False)
print(len(df), "injections, overall recovered:", round(df.recovered.mean(), 2))
df.head()

In [ ]:
inject.plot_completeness(df, period_edges, radius_edges); plt.show()

**Read the map.** Big planets / short periods → bright (found). Small / long → dark (missed).
Why does completeness fall with period? Fewer transits in 27 days ⇒ fewer in-transit points ⇒ lower SNR.

Both effects collapse onto **one** variable: the expected SNR. Plot recovery vs SNR:

In [ ]:
plotting.plot_efficiency(df); plt.show()
s50, width, _ = inject.fit_efficiency_curve(df)
print(f"Your pipeline finds half of all planets with expected SNR = {s50:.1f}")

In [ ]:
df.match_kind.value_counts()     # how did failures fail? 'none' = lost in noise; '2:1' = alias

## 3. Now on real data
Real noise isn't white Gaussian: there are residual systematics, gaps, flares, and the real planet.
**Remove the known planet first**, or your pipeline keeps finding Kepler-10 b instead of the fake one.
(Try it without masking once — it's an instructive failure.)

In [ ]:
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3])
lc.meta.update(rstar=1.065, mstar=0.910)
REAL_PIPE = dict(method="median", window_days=0.5)
# Get the real planet's ephemeris from YOUR OWN search (published ephemerides drift over years)
r0 = search.bls(detrend.flatten(lc, **REAL_PIPE), max_period=10)
known = detrend.transit_mask(lc.time, r0.period, r0.t0, r0.duration, pad=2.0)
lc_clean = LightCurve(lc.time[~known], lc.flux[~known], lc.flux_err[~known], lc.meta)
print(f"known planet at P = {r0.period:.5f} d; removed {known.sum()} in-transit points")
pe = np.geomspace(0.6, 20, 5); re = np.geomspace(0.5, 4, 5)
df_real = inject.injection_recovery(lc_clean, pe, re, n_per_cell=6, detrend_kwargs=REAL_PIPE,
                                    bls_kwargs={"oversample": 1.0, "max_period": 25},
                                    n_jobs=os.cpu_count(), progress=False)
inject.plot_completeness(df_real, pe, re, title="Kepler-10, Q3: detection efficiency"); plt.show()
print("overall:", round(df_real.recovered.mean(), 2))

## 4. The science question: what limits *your* pipeline?
This is where your project becomes *yours*. Pick **one** question, run controlled experiments
(change one thing at a time, same random seed), and answer it with completeness maps.

1. **Detrending window:** run the grid for windows 0.25, 0.5, 1, 2 d. Long-duration (long-period)
   planets should suffer at short windows. Do they? Where's the sweet spot?
2. **Detrending method:** median vs savgol vs biweight. Does Hippke et al.'s conclusion hold for your star?
3. **Detection threshold:** SDE 6 vs 7 vs 9. More planets found — but how many false alarms? (Run the
   pipeline on light curves with *no* injection and count detections above threshold.)
4. **BLS grid:** `oversample` 0.5 vs 1 vs 3. Does a coarse period grid miss planets? Which ones?
5. **Kepler vs TESS:** same planet sizes, a Kepler quarter vs a TESS sector of a similar-brightness star.
6. **Quiet vs active star:** pick a spotted, fast-rotating star. How much completeness do you lose?

### 🚀 Going further (full-timers)
- Scale up: 20+ injections per cell, finer grid, `n_jobs` on a lab machine/cluster. Plot errors on
  each cell: $\sigma_f = \sqrt{f(1-f)/n}$.
- Fit a **gamma CDF** instead of a logistic (Christiansen et al. use this) and compare.
- Turn completeness into an **occurrence rate**: with $N_\text{found}$ planets in a cell from $N_\star$ stars,
  $f_\text{occ} \approx N_\text{found} / (N_\star \cdot \text{completeness} \cdot p_\text{transit})$, where $p_\text{transit}\approx R_\star/a$.
- Inject with `model="batman"` (limb-darkened) instead of trapezoids. Does completeness change?

### ✅ Week 4 Checkpoint (final)
- [ ] Synthetic + real completeness maps committed
- [ ] Efficiency curve + your S50 number
- [ ] One controlled experiment from Section 4 with a clear conclusion
- [ ] Report (4–6 pages) + 5-minute talk / poster — see the project guide, Section 6

In [ ]:
# YOUR EXPERIMENT HERE
# results = {}
# for w in [0.25, 0.5, 1.0, 2.0]:
#     results[w] = inject.injection_recovery(lc_clean, pe, re, n_per_cell=6,
#                       detrend_kwargs=dict(method="median", window_days=w),
#                       bls_kwargs={"oversample": 1.0, "max_period": 25}, seed=0,
#                       n_jobs=os.cpu_count(), progress=False)